In [ ]:
%load ../lib/RecursiveSet.jsh
%load FOL-Parser.jsh

# <a href="https://en.wikipedia.org/wiki/Unification_(computer_science)">Unification</a>

This notebook implements the algorithm of *Martelli and Montanari* for the unification of terms.

## Utility Functions & Strict Typing

We use the record classes for terms that are defined in the file `FOL-Parser.jsh`: A term is either a variable `Var(x)` or a function application `App(f, args)`.  A *syntactical equation* $s \doteq t$ is represented as an object of the record class `Equation`.  A *substitution* is represented as a `Map<String, Term>`.

In [ ]:
// A syntactical equation s ≐ t.
record Equation(Term lhs, Term rhs) {
    public String toString() { return lhs + " ≐ " + rhs; }
}

Terms are represented as trees of records.  In order to convert a string into a term we use the function `parseTerm` that is implemented in the file `FOL-Parser.jsh`.  Our parser distinguishes variables and function symbols as follows:
- A word starting with an upper case letter is interpreted as a *variable*.
- A word starting with a lower case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
// Parses the string s as a term of first-order logic.
Term parseTerm(String s) {
    Object result = new LogicParser(s).parse();
    if (result instanceof Term t) {
        return t;
    }
    throw new IllegalArgumentException("Parsed AST is not a valid term: " + result);
}

In [ ]:
var t0 = parseTerm("f(g(X),Y)");
System.out.println(t0);
System.out.println(structure(t0));

The method $\texttt{apply}(t, \sigma)$ takes an object $t$ and a substitution $\sigma$ and computes the $t\sigma$, i.e. it *applies* the substitution $\sigma$ to $t$.  The object $t$ is either a term, a *syntactic equation*, or a list of syntactic equations.  The substitution $\sigma$ is represented as a map.  Assume that $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$.  Then $t\sigma$ is defined by induction on $t$:
- If $t$ is a variable, there are two cases when defining $t\sigma$:
  - $t = x_i$ for an $i\in\{1,\cdots,n\}$.  Then we define  
    $$ x_i\sigma := t_i. $$
  - $t = y$ where $y\in\mathcal{V}$, but $y \not\in \{x_1,\cdots,x_n\}$. Then we define   
    $$ y\sigma := y.$$</li>
- Otherwise, we must have $t = f(s_1,\cdots,s_m)$. Then we define: 
  $$ f(s_1, \cdots, s_m)\sigma := f(s_1\sigma, \cdots, s_m\sigma). $$

As Java has no union types, there are three different functions: `applyTerm` applies a substitution to a term, `applyEquation` applies it to an equation, and `applyEquations` applies it to a list of equations.

In [ ]:
// Apply the substitution sigma, which maps variables to terms, to the term t.
Term applyTerm(Term t, Map<String, Term> sigma) {
    return switch (t) {
        case Var(var x)           -> sigma.getOrDefault(x, t);
        case App(var f, var args) -> new App(f, args.stream().map(arg -> applyTerm(arg, sigma)).toList());
        default -> throw new IllegalArgumentException("unknown term: " + t);
    };
}

In [ ]:
Equation applyEquation(Equation eq, Map<String, Term> sigma) {
    return new Equation(applyTerm(eq.lhs(), sigma), applyTerm(eq.rhs(), sigma));
}

In [ ]:
List<Equation> applyEquations(List<Equation> E, Map<String, Term> sigma) {
    return E.stream().map(eq -> applyEquation(eq, sigma)).toList();
}

In [ ]:
var s1 = parseTerm("g(Z)");
var s2 = parseTerm("h(U, V)");

Map<String, Term> sigma = new LinkedHashMap<>();
sigma.put("X", s1);
sigma.put("Y", s2);
sigma

In [ ]:
var tTerm = parseTerm("f(X,h(Y,X),g(Z))");
tTerm

In [ ]:
applyTerm(tTerm, sigma)

If  $\sigma = \big\{ x_1 \mapsto s_1, \cdots, x_m \mapsto s_m \big\}$ and
$\tau = \big\{ y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$ 
are two substitutions that are <em style="color:blue;">non-overlapping</em>, i.e. such that $\texttt{dom}(\sigma) \cap \texttt{dom}(\tau) = \{\}$ holds,
then we define the <em style="color:blue;">composition</em> $\sigma\tau$ of $\sigma$ and $\tau$ as follows:
$$\sigma\tau := \big\{ x_1 \mapsto s_1\tau, \cdots, x_m \mapsto s_m\tau,\; y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$$
The function $\texttt{compose}(\sigma, \tau)$ takes two non-overlapping substitutions and computes the composition $\sigma\tau$.

In [ ]:
// The composition of the non-overlapping substitutions sigma and tau.
Map<String, Term> compose(Map<String, Term> sigma, Map<String, Term> tau) {
    Map<String, Term> result = new LinkedHashMap<>();
    for (var entry : sigma.entrySet()) {
        result.put(entry.getKey(), applyTerm(entry.getValue(), tau));
    }
    result.putAll(tau);
    return result;
}

In [ ]:
Map<String, Term> tau = new LinkedHashMap<>();
tau.put("Z", s1);
tau.put("U", s2);

System.out.println("Sigma: " + sigma);
System.out.println("Tau: " + tau);
System.out.println("Composed: " + compose(sigma, tau));

The function $\texttt{occurs}(x, t)$ checks whether the variable $x$ occurs in the term $t$.

In [ ]:
// Checks whether the variable x occurs in the term t.
boolean occurs(String x, Term t) {
    return switch (t) {
        case Var(var y)           -> x.equals(y);
        case App(var _, var args) -> args.stream().anyMatch(arg -> occurs(x, arg));
        default -> throw new IllegalArgumentException("unknown term: " + t);
    };
}

In [ ]:
System.out.println(tTerm);
System.out.println("Occurs 'U': " + occurs("U", tTerm));
System.out.println("Occurs 'X': " + occurs("X", tTerm));

## The Algorithm of Martelli and Montanari

The rules of Martelli and Montanari that can be used to solve a system of syntactical equations are as follows:
<ol>
<li> If $y\in\mathcal{V}$ is a variable that does <b style="color:red;">not</b> occur in the term $t$,
     then we perform the following reduction: 
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle \quad\leadsto \quad 
         \Big\langle E\{y \mapsto t\}, \sigma\big\{ y \mapsto t \big\} \Big\rangle 
     $$
</li>      
<li> If the variable $y$ occurs in the term $t$, then the system of syntactical equations
     $E \cup \big\{ y \doteq t \big\}$ is not solvable:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle\;\leadsto\; \Omega \quad
        \mbox{if $y \in \textrm{Var}(t)$ and $y \not=t$.}$$
</li>
<li> If $y\in\mathcal{V}$ is a variable and $t$ is no variable, then we use the following rule:
     $$ \Big\langle E \cup \big\{ t \doteq y \big\}, \sigma \Big\rangle \quad\leadsto \quad 
         \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle.
     $$   
</li>
<li> Trivial syntactical equations of variables can be dropped:
     $$ \Big\langle E \cup \big\{ x \doteq x \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E, \sigma \Big\rangle.
     $$   
</li>
<li> If $f$ is an $n$-ary function symbol, then we have: 
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_n) \doteq f(t_1,\cdots,t_n) \big\}, \sigma \Big\rangle 
         \;\leadsto\; 
         \Big\langle E \cup \big\{ s_1 \doteq t_1, \cdots, s_n \doteq t_n\}, \sigma \Big\rangle.
     $$   
</li>
<li> The system of syntactical equations $E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\}$
     has <b style="color:red;">no</b> solution if the function symbols $f$ and $g$ are different:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\},
      \sigma \Big\rangle \;\leadsto\; \Omega \qquad \mbox{if $f \not= g$}.
     $$
</ol>


Given a list of <em style="color:blue;">syntactical equations</em> $E$ and a substitution $\sigma$, the function $\texttt{solve}(E, \sigma)$ uses the rules of Martelli and Montanari to solve $E$ recursively.  If $E$ is not solvable, the function returns `null`.  As Java allows Greek letters in identifiers, we can use the names `σ` and `τ` for substitutions.

Note that two terms are compared with the method `equals`, which compares records structurally.

In [ ]:
// Solve the list of syntactical equations E using the rules of Martelli and Montanari.
// σ is the substitution computed so far.  Returns null if E is not solvable.
Map<String, Term> solve(List<Equation> E, Map<String, Term> σ) {
    if (E.isEmpty()) {
        return σ;
    }
    var eq    = E.get(0);
    var restE = E.subList(1, E.size());
    Term s = eq.lhs();
    Term t = eq.rhs();
    if (s.equals(t)) {
        return solve(restE, σ);
    }
    if (s instanceof Var(var x)) {
        if (occurs(x, t)) {
            return null;
        }
        Map<String, Term> τ = Map.of(x, t);
        return solve(applyEquations(restE, τ), compose(σ, τ));
    }
    if (t instanceof Var) {
        List<Equation> newE = new ArrayList<>();
        newE.add(new Equation(t, s));
        newE.addAll(restE);
        return solve(newE, σ);
    }
    var sApp = (App) s;
    var tApp = (App) t;
    if (!sApp.function().equals(tApp.function()) || sApp.args().size() != tApp.args().size()) {
        return null;
    }
    List<Equation> newE = new ArrayList<>();
    for (int i = 0; i < sApp.args().size(); i++) {
        newE.add(new Equation(sApp.args().get(i), tApp.args().get(i)));
    }
    newE.addAll(restE);
    return solve(newE, σ);
}

Given two terms $s$ and $t$, the function $\texttt{unify}(s, t)$ computes the <em style="color:blue;">most general unifier</em> of $s$ and $t$.

In [ ]:
// The most general unifier of the terms s and t.  Returns null if s and t are not unifiable.
Map<String, Term> unify(Term s, Term t) {
    return solve(List.of(new Equation(s, t)), new LinkedHashMap<>());
}

In [ ]:
var t1 = parseTerm("p(X1,f(X1))");
var t2 = parseTerm("p(g(X2),X3)");
List.of(t1, t2)

In [ ]:
var mu = unify(t1, t2);
mu

The occurs check prevents the unification of a variable with a term that contains this variable.  Hence the following call returns `null`.

In [ ]:
System.out.println(unify(parseTerm("f(X)"), parseTerm("f(g(X))")));

The functions defined in this notebook are also available in the file `Unification.jsh`.  This file is used in the notebook `03-Prover.ipynb`.